# Notebook 05 — Exploratory Business & Commercial Analysis

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 7 — Exploratory Data Analysis & Business Intelligence  

## Analytical Domains Covered
1. **Executive Revenue & Growth:** Monthly GMV trajectory, order volume dynamics, AOV.
2. **Seller Performance Matrix:** Strategic 4-Quadrant analysis (Stars, Risks, Champions, Underperformers).
3. **Product & Category Dynamics:** Pareto 80/20 category concentration, freight ratio friction.
4. **Customer Retention & Financing:** Repeat buyer economics, credit card installments, payment instruments.

---
## 0. Setup & Data Ingestion

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DB_PATH = '../data/processed/ecommerce_control_tower.db'
conn = sqlite3.connect(DB_PATH)

fact_orders = pd.read_sql_query('SELECT * FROM fact_orders;', conn)
fact_items = pd.read_sql_query('SELECT * FROM fact_order_items;', conn)
dim_customers = pd.read_sql_query('SELECT * FROM dim_customers;', conn)
dim_sellers = pd.read_sql_query('SELECT * FROM dim_sellers;', conn)
dim_products = pd.read_sql_query('SELECT * FROM dim_products;', conn)
fact_payments = pd.read_sql_query('SELECT * FROM fact_payments;', conn)

fact_orders['order_purchase_timestamp'] = pd.to_datetime(fact_orders['order_purchase_timestamp'])
print(f'Successfully loaded facts & dimensions from {DB_PATH}')

---
## 1. Executive Revenue & Order Trajectory

In [ ]:
# Monthly GMV and Order Volume Trajectory
monthly = fact_orders.set_index('order_purchase_timestamp').resample('ME').agg(
    orders=('order_id', 'count'),
    gmv=('gmv', 'sum'),
    freight=('freight_value', 'sum'),
    aov=('gmv', 'mean')
).reset_index()

# Filter core operating window (Jan 2017 to Aug 2018)
monthly = monthly[(monthly['order_purchase_timestamp'] >= '2017-01-01') & (monthly['order_purchase_timestamp'] <= '2018-08-31')]
monthly['ym'] = monthly['order_purchase_timestamp'].dt.strftime('%b %Y')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly['ym'], monthly['gmv'], color='#1f77b4', marker='o', linewidth=2.5, label='GMV (R$)')
ax1.set_ylabel('Gross Merchandise Value (R$)', color='#1f77b4', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_xticks(range(len(monthly['ym'])))
ax1.set_xticklabels(monthly['ym'], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.bar(monthly['ym'], monthly['orders'], color='#ff7f0e', alpha=0.3, width=0.4, label='Orders')
ax2.set_ylabel('Total Order Volume', color='#ff7f0e', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#ff7f0e')
ax2.grid(False)

plt.title('Monthly GMV and Order Volume Growth (2017–2018)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 2. Seller Strategic 4-Quadrant Matrix

In [ ]:
# Filter active sellers (>= 10 orders)
sellers = dim_sellers[dim_sellers['total_orders'] >= 10].copy()
gmv_median = sellers['total_gmv'].median()
csat_threshold = 4.0

def classify_seller(row):
    if row['total_gmv'] >= gmv_median and row['avg_review_score'] >= csat_threshold:
        return 'Star Performers (High GMV / High CSAT)'
    elif row['total_gmv'] >= gmv_median and row['avg_review_score'] < csat_threshold:
        return 'Operational Risk (High GMV / Low CSAT)'
    elif row['total_gmv'] < gmv_median and row['avg_review_score'] >= csat_threshold:
        return 'Niche Champions (Low GMV / High CSAT)'
    else:
        return 'Underperformers (Low GMV / Low CSAT)'

sellers['quadrant'] = sellers.apply(classify_seller, axis=1)

print('Seller Quadrant Distribution:')
print(sellers['quadrant'].value_counts())

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Star Performers (High GMV / High CSAT)': '#2ca02c', 'Operational Risk (High GMV / Low CSAT)': '#d62728', 'Niche Champions (Low GMV / High CSAT)': '#1f77b4', 'Underperformers (Low GMV / Low CSAT)': '#7f7f7f'}
for q_name, grp in sellers.groupby('quadrant'):
    ax.scatter(grp['total_gmv'], grp['avg_review_score'], label=f'{q_name} (n={len(grp)})', color=colors[q_name], alpha=0.6, s=grp['total_orders']*0.8+15)

ax.axvline(gmv_median, color='black', linestyle=':')
ax.axhline(csat_threshold, color='black', linestyle=':')
ax.set_xscale('log')
ax.set_title('Seller Strategic 4-Quadrant Matrix (GMV vs CSAT)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Seller Total GMV (R$, Log Scale)', fontweight='bold')
ax.set_ylabel('Average Review Score (1 to 5)', fontweight='bold')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

---
## 3. Product & Category Pareto Dynamics

In [ ]:
# Category GMV and Pareto Cumulative Contribution
cat_summary = fact_items.groupby('product_category_name_english').agg(
    total_gmv=('price', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_freight_ratio=('freight_ratio_pct', 'mean'),
    avg_item_price=('price', 'mean')
).reset_index().sort_values('total_gmv', ascending=False)

cat_summary['cumulative_gmv'] = cat_summary['total_gmv'].cumsum()
cat_summary['cumulative_pct'] = (cat_summary['cumulative_gmv'] / cat_summary['total_gmv'].sum()) * 100

# Top 10 categories
print('Top 10 Categories by GMV & Pareto Contribution:')
cat_summary.head(10)[['product_category_name_english', 'total_gmv', 'cumulative_pct', 'avg_freight_ratio']]

---
## 4. Payment Economics & Customer Retention

In [ ]:
# Payment Method Breakdown
pay_stats = fact_payments.groupby('payment_type').agg(
    total_value=('payment_value', 'sum'),
    transactions=('order_id', 'count'),
    avg_installments=('payment_installments', 'mean')
).reset_index()
pay_stats['pct_of_total_value'] = (pay_stats['total_value'] / pay_stats['total_value'].sum()) * 100
pay_stats.sort_values('total_value', ascending=False)